In [1]:
import pandas as pd

tokenizers and embeddings are important components to language models. working in the textual domain is difficult - machine learning models generally do not ingest raw text. instead, text is turned into *tokens*, which are essentially unique ids for specific words (or parts of words), and then turned into *embeddings*, where tokens are projected onto some real vector space. the neural network uses this vector space as it's input (representing that word or that part of the word), and then it's output is in the same dimension, which is then backed out into a word.

we first try using the `gpt-2` tokenizer to understand how this works.

In [5]:
from transformers import AutoTokenizer

tokeniser = AutoTokenizer.from_pretrained("gpt2")
text = "The brown fox jumped over the lazy dog"

as you can see below, the words are converted into unique ids, which can then be transferred *back into tokens*. so below we are seeing the first and last layers/steps of our neural network (although we do not need to train/create our tokenizer).

In [7]:
ids = tokeniser.encode(text)
print(f'IDs: {ids}')
print(tokeniser.convert_ids_to_tokens(ids))

IDs: [464, 7586, 21831, 11687, 625, 262, 16931, 3290]
['The', 'Ġbrown', 'Ġfox', 'Ġjumped', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog']


all a tokenizer really does is provide a very big mapping from `word`s in a vocabulary into an `index`, guaranteed to be unique. we can then declaring an embedding matrix $\mathbf{M} \in \mathbb{R}^{T \times M}$ for vocabulary size $T$ and dimensions in embedding $M$. Thus the first row of $M$ will represent an embedding for the first token ID and so on. this embedding can be trained simultaneously during back propagation, which would then in-turn learn to separate the words.

In [ ]:
import torch
import torch.nn as nn

tokeniser.vocab_size
# size [token vocabulary * embedding dimension]
embedding_mat = nn.Embedding(num_embeddings=tokeniser.vocab_size, embedding_dim=512)

then, by indexing on a torch tensor with a single value representing the id, we should get an embedding of dimension $\mathbb{R}^{512 \times 1}$.

In [14]:
embedding_mat(torch.tensor([1]))

tensor([[-1.8614e-01, -2.3268e+00, -1.9101e-01, -9.8703e-01,  7.0271e-01,
          2.4295e+00, -6.8812e-01,  8.7858e-01, -2.1454e-01, -2.4935e-01,
         -7.5453e-02,  1.6525e+00, -1.3796e+00, -1.7506e+00,  2.2899e-01,
          1.0080e+00,  6.2780e-01,  2.9585e-01, -1.5387e+00,  1.2993e+00,
         -1.5265e-01, -1.7678e-01, -1.4278e+00, -8.3659e-01,  2.5722e-02,
          9.1005e-01, -1.8709e+00,  1.1318e+00, -1.5607e-01, -1.0675e+00,
         -1.2911e+00,  6.0861e-01,  1.2267e+00,  2.0061e+00,  2.8647e-01,
          1.3115e+00, -6.9822e-01, -5.6981e-01,  3.8822e-01, -7.0308e-01,
          6.1689e-01, -8.7958e-02,  4.9639e-01,  3.1242e-02, -2.5367e+00,
          2.2122e+00,  2.9784e-01, -2.2694e-01,  1.3877e+00, -2.3691e-01,
         -2.6352e-01, -2.5019e-01,  1.2298e+00,  7.5451e-01,  8.6179e-01,
         -9.4413e-03, -2.5055e-01,  1.4905e+00, -5.5032e-01, -6.4065e-01,
          3.3683e-01, -2.2290e+00, -1.6640e+00, -8.1307e-01,  1.1178e-01,
          1.1798e+00, -4.9619e-01, -3.

at the output of our neural network, we want to convert the networks embedding vector back *into* a token id. so the matrix prior to the output should likely be of dimension $\mathbb{R}^{n \times 512}$, and then the output layer would be $\mathbb{R}^{512 \times \text{vocab size}}$. then softmax can be used on that output layer to get the most likely word.